# HyperRAG Deep Dive -- Why Graph Expansion Finds No Neighbors

This notebook diagnoses why `hyperrag()` produces **identical results to `naive_rag()`** despite having a graph expansion step.

HyperRAG is supposed to:
1. Retrieve top-k pages via FAISS (same as Naive RAG)
2. Expand via 1-hop hyperlink graph neighbors
3. Re-rank and append the best neighbors to the context

We will show that step 2 finds **zero or very few neighbors**, making the output identical to Naive RAG.

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# Ensure project root is on path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.corpus import load_corpus, load_hotpotqa
from src.embeddings import load_index, search, build_faiss_index
from src.graph import load_graph
from src.retrieval import hyperrag, naive_rag, compute_em, compute_f1

# Load data
corpus = load_corpus(PROJECT_ROOT / "data" / "corpus.json")
index, page_ids = load_index(PROJECT_ROOT / "data")
graph = load_graph(PROJECT_ROOT / "data" / "hyperlink_graph.graphml")
qa_items = load_hotpotqa(split="train", n_samples=50)[:10]

print(f"Corpus: {len(corpus)} pages")
print(f"FAISS index: {index.ntotal} vectors")
print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print(f"QA items loaded: {len(qa_items)}")

## Step 1: Graph Structure Overview

The hyperlink graph connects Wikipedia pages that link to each other **within** the corpus.
Since the corpus is only ~450 pages (supporting pages from 500 HotpotQA questions), this is a very sparse graph.

Let's examine the structure.

In [ ]:
n_nodes = graph.number_of_nodes()
n_edges = graph.number_of_edges()

out_degrees = [d for _, d in graph.out_degree()]
in_degrees = [d for _, d in graph.in_degree()]
total_degrees = [d for _, d in graph.degree()]

avg_out = np.mean(out_degrees) if out_degrees else 0
avg_in = np.mean(in_degrees) if in_degrees else 0
isolated = sum(1 for d in total_degrees if d == 0)

print(f"Nodes: {n_nodes}")
print(f"Edges: {n_edges}")
print(f"Average out-degree: {avg_out:.2f}")
print(f"Average in-degree: {avg_in:.2f}")
print(f"Isolated nodes (degree 0): {isolated}")
print(f"Corpus pages: {len(corpus)}")
print(f"Nodes vs corpus: {n_nodes} vs {len(corpus)}")
print()

# Top-5 highest degree nodes
top5 = sorted(graph.degree(), key=lambda x: x[1], reverse=True)[:5]
print("Top-5 highest degree nodes:")
for rank, (title, deg) in enumerate(top5, 1):
    print(f"  {rank}. {title} (degree={deg})")
print()

# Degree distribution histogram
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(total_degrees, bins=range(0, max(total_degrees) + 2), color="steelblue",
        edgecolor="black", alpha=0.7)
ax.set_xlabel("Degree (in + out)")
ax.set_ylabel("Count (number of nodes)")
ax.set_title(f"Degree Distribution ({n_nodes} nodes, {n_edges} edges)")
ax.axvline(np.mean(total_degrees), color="red", linestyle="--",
           label=f"Mean degree = {np.mean(total_degrees):.1f}")
ax.legend()
plt.tight_layout()
plt.show()

## Step 2: Node ID Matching -- The Critical Check

For graph expansion to work, the page titles used as node IDs in the graph must **exactly match** the titles in the corpus. After saving to GraphML and loading back, there may be encoding mismatches (special characters, HTML entities, unicode normalization).

This is the **most important diagnostic cell** in this notebook.

In [ ]:
graph_nodes = set(graph.nodes())
corpus_titles = set(p["title"] for p in corpus)

intersection = graph_nodes & corpus_titles
graph_only = graph_nodes - corpus_titles
corpus_only = corpus_titles - graph_nodes

print(f"Graph nodes:     {len(graph_nodes)}")
print(f"Corpus titles:   {len(corpus_titles)}")
print(f"Intersection:    {len(intersection)} (nodes found in both)")
print(f"Graph only:      {len(graph_only)} (in graph but NOT in corpus titles)")
print(f"Corpus only:     {len(corpus_only)} (in corpus but NOT in graph nodes)")
print()

if graph_only:
    print("First 5 graph-only nodes (possible encoding mismatches):")
    for t in list(graph_only)[:5]:
        print(f"  Graph: {repr(t)}")
    print()

if corpus_only:
    print("First 5 corpus-only titles (not found in graph):")
    for t in list(corpus_only)[:5]:
        print(f"  Corpus: {repr(t)}")
    print()

# Check specific titles with special characters
special_char_titles = [t for t in corpus_titles if any(c in t for c in '&\'"()[]')]
if special_char_titles:
    print(f"Checking {min(5, len(special_char_titles))} corpus titles with special chars:")
    for t in special_char_titles[:5]:
        found = graph.has_node(t)
        print(f"  '{t}' -> graph.has_node() = {found}")
    print()

pct = len(intersection) / len(corpus_titles) * 100 if corpus_titles else 0
print(f"DIAGNOSIS: {len(intersection)} out of {len(corpus_titles)} corpus titles "
      f"are found as graph nodes ({pct:.1f}%)")

## Step 3: Graph Expansion for Retrieved Pages

Now let's trace the HyperRAG algorithm step by step for a single question.

The algorithm:
1. Retrieve top-5 pages via FAISS (same as Naive RAG)
2. For each retrieved page, check if it exists in the graph
3. If it does, collect successors (out-links) and predecessors (in-links)
4. Filter to within-corpus neighbors only
5. Exclude pages already in the initial set

If no neighbors are found, HyperRAG returns the **exact same context** as Naive RAG.

In [ ]:
qa = qa_items[0]
question = qa["question"]
answer = qa["answer"]

print(f"Question: {question}")
print(f"Answer: {answer}")
print()

# Step 1: Initial FAISS retrieval (same as naive_rag)
initial_pages = search(question, index, page_ids, corpus, k=5)
initial_titles = set(p["title"] for p in initial_pages)
corpus_title_set = set(p["title"] for p in corpus)

print("Initial FAISS retrieval (same as Naive RAG):")
all_neighbors = set()

for p in initial_pages:
    title = p["title"]
    has_node = graph.has_node(title)
    print(f"\n  Page: '{title}'")
    print(f"    graph.has_node() = {has_node}")

    if has_node:
        successors = list(graph.successors(title))
        predecessors = list(graph.predecessors(title))
        print(f"    Successors (out-links): {len(successors)}")
        print(f"    Predecessors (in-links): {len(predecessors)}")

        # Filter to within-corpus and exclude initial pages
        succ_in_corpus = [s for s in successors if s in corpus_title_set and s not in initial_titles]
        pred_in_corpus = [p for p in predecessors if p in corpus_title_set and p not in initial_titles]
        print(f"    Successors in corpus (excl. initial): {len(succ_in_corpus)}")
        print(f"    Predecessors in corpus (excl. initial): {len(pred_in_corpus)}")

        if succ_in_corpus:
            print(f"      -> {succ_in_corpus[:3]}")
        if pred_in_corpus:
            print(f"      -> {pred_in_corpus[:3]}")

        all_neighbors.update(succ_in_corpus)
        all_neighbors.update(pred_in_corpus)
    else:
        print("    SKIPPED: node not found in graph")

print(f"\n{'=' * 60}")
print(f"Total unique neighbors found: {len(all_neighbors)}")
if all_neighbors:
    print(f"Neighbor titles: {list(all_neighbors)[:10]}")
else:
    print("NO NEIGHBORS FOUND -- HyperRAG returns SAME context as Naive RAG")

## Step 4: Subgraph Visualization

Let's visualize the subgraph around the retrieved pages.
- **Blue nodes:** Initial FAISS-retrieved pages
- **Green nodes:** Graph neighbors (if any)
- **Gray edges:** Hyperlinks between nodes

If the graph is sparse, you will see mostly disconnected blue nodes with few or no green neighbors.

In [ ]:
# Collect nodes for the subgraph
nodes_to_show = set(initial_titles)
# Add neighbors (limit to 15 to keep visualization readable)
neighbors_to_show = list(all_neighbors)[:15]
nodes_to_show.update(neighbors_to_show)

# Only include nodes that actually exist in the graph
valid_nodes = [n for n in nodes_to_show if graph.has_node(n)]

if len(valid_nodes) < 2:
    print(f"Only {len(valid_nodes)} valid graph nodes -- adding some connected nodes for context")
    # Add a few high-degree nodes from the graph for context
    top_nodes = sorted(graph.degree(), key=lambda x: x[1], reverse=True)[:10]
    for node, deg in top_nodes:
        valid_nodes.append(node)
        if len(valid_nodes) >= 15:
            break

subgraph = graph.subgraph(valid_nodes)

# Color coding
node_colors = []
for n in subgraph.nodes():
    if n in initial_titles:
        node_colors.append("steelblue")
    elif n in all_neighbors:
        node_colors.append("lightgreen")
    else:
        node_colors.append("lightgray")

# Shorten labels for readability
labels = {n: n[:25] + "..." if len(n) > 25 else n for n in subgraph.nodes()}

fig, ax = plt.subplots(figsize=(12, 8))
pos = nx.spring_layout(subgraph, k=2, seed=42)
nx.draw_networkx(
    subgraph, pos, ax=ax,
    node_color=node_colors,
    edge_color="gray",
    font_size=6,
    node_size=500,
    labels=labels,
    arrows=True,
    arrowsize=10,
    alpha=0.9
)
ax.set_title("Subgraph around retrieved pages\n(blue=initial FAISS, green=graph neighbors, gray=context nodes)")
plt.tight_layout()
plt.show()

print(f"Subgraph: {subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges")

## Step 5: Full Pipeline Comparison -- With vs Without Graph Expansion

Now let's run all 10 questions through both Naive RAG and HyperRAG, comparing:
- EM and F1 scores
- Context length (if graph expansion works, HyperRAG context should be longer)

In [ ]:
comparison = []

for i, qa in enumerate(qa_items):
    q = qa["question"]
    a = qa["answer"]

    ctx_naive = naive_rag(q, index, corpus, k=5)
    ctx_hyper = hyperrag(q, index, corpus, graph, k=5, expand_k=3)

    n_em = compute_em(ctx_naive, a)
    h_em = compute_em(ctx_hyper, a)
    n_f1 = compute_f1(ctx_naive, a)
    h_f1 = compute_f1(ctx_hyper, a)

    comparison.append({
        "qid": i, "naive_em": n_em, "hyper_em": h_em,
        "naive_f1": n_f1, "hyper_f1": h_f1,
        "naive_len": len(ctx_naive), "hyper_len": len(ctx_hyper),
        "len_diff": len(ctx_hyper) - len(ctx_naive)
    })

# Print comparison table
print(f"{'QID':<5} {'N_EM':<6} {'H_EM':<6} {'N_F1':<10} {'H_F1':<10} {'N_Len':<8} {'H_Len':<8} {'Diff'}")
print("-" * 70)
for c in comparison:
    print(f"{c['qid']:<5} {c['naive_em']:<6.0f} {c['hyper_em']:<6.0f} "
          f"{c['naive_f1']:<10.6f} {c['hyper_f1']:<10.6f} "
          f"{c['naive_len']:<8} {c['hyper_len']:<8} {c['len_diff']}")

differ_count = sum(1 for c in comparison if c["naive_f1"] != c["hyper_f1"])
print(f"\nQuestions where HyperRAG differs from Naive RAG: {differ_count}/{len(comparison)}")

len_diff_count = sum(1 for c in comparison if c["len_diff"] > 0)
print(f"Questions where HyperRAG context is LONGER (graph expansion worked): {len_diff_count}/{len(comparison)}")

# Grouped bar chart: Naive vs HyperRAG F1
x = np.arange(len(comparison))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, [c["naive_f1"] for c in comparison], width,
       label="Naive RAG", color="steelblue")
ax.bar(x + width/2, [c["hyper_f1"] for c in comparison], width,
       label="HyperRAG", color="coral")

ax.set_xlabel("Question Index")
ax.set_ylabel("F1 Score")
ax.set_title("Naive RAG vs HyperRAG: F1 per Question (10 questions)")
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.show()

## Diagnosis

### Flaw 1: Shared Initial Retrieval

HyperRAG starts with the **exact same FAISS search** as Naive RAG. The graph expansion is purely **additive** -- it can only ADD pages on top of the initial retrieval, never replace them. This means HyperRAG can only be as good or better than Naive RAG, never worse.

### Flaw 3: Graph Expansion Failure

The hyperlink graph is built from **within-corpus links only** (~450 pages). This creates a very sparse graph:

1. **Sparse graph:** With only ~450 nodes, most pages link to pages OUTSIDE the corpus. Within-corpus edges are rare because Wikipedia has millions of pages but our corpus is tiny.

2. **Node ID encoding mismatches:** After saving to GraphML and loading back, node IDs may have encoding differences (HTML entities, unicode normalization). If `graph.has_node(title)` returns False for a corpus title, that page gets zero expansion.

3. **Empty neighbor sets:** When no within-corpus neighbors are found, `top_neighbors` is empty, and `hyperrag()` returns `initial_pages + []` -- the exact same context as `naive_rag()`.

### Evidence from This Notebook

- **Cell 6 (Node ID Matching):** Shows how many corpus titles actually match graph nodes
- **Cell 8 (Neighbor Expansion):** Shows per-page neighbor counts -- most pages have 0 within-corpus neighbors
- **Cell 12 (Pipeline Comparison):** Shows `len_diff = 0` for questions where graph expansion found nothing

### What Would Fix It

1. **Larger corpus = denser graph:** With 5,000+ pages instead of ~450, there would be far more within-corpus links. The graph would have higher average degree, and expansion would actually find relevant bridge pages.

2. **Fix node ID encoding mismatches:** Ensure that GraphML save/load preserves title encoding exactly. Use `html.unescape()` consistently on both corpus titles and graph node IDs.

3. **Use 2-hop expansion:** Instead of only looking at direct neighbors, traverse 2 hops to reach more pages. This squares the neighborhood size.

4. **Add external link expansion:** When a retrieved page links to a page NOT in the corpus, fetch that page on-the-fly and add it to the context. This would dramatically increase coverage but adds latency.